1. importing os and pandas for file and data manipulation

In [47]:
import os
import pandas as pd

2. 

In [48]:
os.makedirs("../data/cleaned", exist_ok=True)

3.

In [49]:
inventory_clean = pd.read_csv("../data/raw/inventory_raw.csv")

In [50]:
print(inventory_clean.shape)
print(inventory_clean.isnull().sum())
print("negative unit sold\n", (inventory_clean["units_sold"]<0).sum())
print(inventory_clean["warehouse"].value_counts())


(50000, 10)
inventory_id        0
snapshot_date       0
product_id          0
warehouse           0
opening_stock       0
units_received      0
units_sold          0
damaged_units       0
closing_stock     250
stock_status        0
dtype: int64
negative unit sold
 200
warehouse
Bengaluru    10132
Kolkata      10093
Delhi         9972
Mumbai        9923
Hyderabad     9880
Name: count, dtype: int64


4. clean text column and standardize category

In [51]:
text_cols = ["inventory_id",
            "product_id",
            "warehouse",
            "stock_status",
            ]
for col in text_cols:
    inventory_clean[col] = inventory_clean[col].astype(str).str.strip()

5. Standardize casing for categorical columns

In [52]:
inventory_clean["warehouse"] = inventory_clean["warehouse"].str.title()
inventory_clean["stock_status"] = inventory_clean["stock_status"].str.title()

6. Convert negative units sold to positive

In [53]:
inventory_clean["units_sold"]  = inventory_clean["units_sold"].abs()

7. Impute Missing Closing Stock Using Formula

Formula: closing_stock = opening_stock + units_received - units_sold - damaged_units

In [54]:
stock_cols = [
    "opening_stock",
    "units_received",
    "units_sold",
    "damaged_units",
    "closing_stock"
]

for col in stock_cols:
    inventory_clean[col] = pd.to_numeric(
        inventory_clean[col],
        errors="coerce"
    )

8. 

In [55]:
calculated_closing = (
    inventory_clean["opening_stock"]
    + inventory_clean["units_received"]
    - inventory_clean["units_sold"]
    - inventory_clean["damaged_units"]
)

inventory_clean["closing_stock"] = inventory_clean["closing_stock"].fillna(
    calculated_closing
)

9. Datetime Parsing

In [56]:
inventory_clean["snapshot_date"] = pd.to_datetime(
    inventory_clean["snapshot_date"], errors="coerce"
)

10. Verify Cleaning Results

In [57]:
print("Dataset Shape:", inventory_clean.shape)
print("Remaining Missing Values:\n", inventory_clean.isnull().sum())
print(
    "Remaining Negative Units Sold:", (inventory_clean["units_sold"] < 0).sum()
)
print("Stock Status Categories:\n", inventory_clean["stock_status"].value_counts())

Dataset Shape: (50000, 10)
Remaining Missing Values:
 inventory_id      0
snapshot_date     0
product_id        0
warehouse         0
opening_stock     0
units_received    0
units_sold        0
damaged_units     0
closing_stock     0
stock_status      0
dtype: int64
Remaining Negative Units Sold: 0
Stock Status Categories:
 stock_status
Overstocked    43579
Healthy         3258
Critical        2246
Low              917
Name: count, dtype: int64


11. Export Cleaned Dataset to '../data/cleaned/' Directory

In [58]:
output_path = os.path.join("../data/cleaned", "inventory_cleaned.csv")
inventory_clean.to_csv(output_path, index=False)